In [1]:
using Ferrite
using ModularEIT
using Images
using IterativeSolvers
using LinearAlgebra
using Plots
using Distributions
using Statistics
using Lux
using JLD2

In [2]:
η = 0.15

σf = 0.0 # how much noise is added to f
σg = 0.0 # How much noise is added to g
# Steers prox operators:
ρ_obj = 1.0e-3

0.001

In [3]:
img = load("../reconstructions/1096.jpg")
img_f = img .|> Float64
img_f .-= 0.5
img_f .*= 2.0
img_f .= exp.(img_f) # Just an idea maybe EIT reconstruction is easier with this.

150×150 Matrix{Float64}:
 1.38471   1.39561   1.4066    1.4066    …  1.89499   1.88019   1.88019
 1.41768   1.42884   1.44009   1.44009      1.90992   1.90992   1.90992
 1.44009   1.45143   1.46286   1.46286      1.92495   1.92495   1.92495
 1.46286   1.47438   1.48599   1.48599      1.92495   1.92495   1.92495
 1.48599   1.49769   1.50948   1.50948      1.94011   1.94011   1.94011
 1.47438   1.48599   1.49769   1.49769   …  1.97078   1.97078   1.97078
 1.44009   1.45143   1.46286   1.46286      1.9863    1.9863    1.9863
 1.44009   1.45143   1.46286   1.46286      1.97078   1.9863    1.9863
 1.46286   1.46286   1.47438   1.48599      2.01771   2.00194   1.9863
 1.47438   1.48599   1.48599   1.49769      2.00194   2.00194   1.9863
 ⋮                                       ⋱                      
 0.367879  0.382593  0.388641  0.407368     0.388641  0.373696  0.388641
 0.394786  0.437162  0.385605  0.385605     0.394786  0.367879  0.373696
 0.385605  0.417066  0.42366   0.417066     0.39

In [4]:
itp = interpolate_array_2D(Float64.(img_f))

#interpolate_array_2D##0 (generic function with 1 method)

In [5]:
n = 63 
grid = generate_grid(Quadrilateral, (n, n));

In [6]:
∂Ω = union(getfacetset.((grid,), ["left", "top", "right", "bottom"])...)
fe  = FerriteFESpace{RefQuadrilateral}(grid,1,0,3,∂Ω)

FerriteFESpace{RefQuadrilateral}(CellValues{Ferrite.FunctionValues{1, Lagrange{RefQuadrilateral, 1}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Matrix{Vec{2, Float64}}, Nothing, Nothing}, Ferrite.GeometryMapping{1, Lagrange{RefQuadrilateral, 1}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Nothing}, QuadratureRule{RefQuadrilateral, Vector{Float64}, Vector{Vec{2, Float64}}}, Vector{Float64}}(Ferrite.FunctionValues{1, Lagrange{RefQuadrilateral, 1}, Matrix{Float64}, Matrix{Vec{2, Float64}}, Matrix{Vec{2, Float64}}, Nothing, Nothing}(Lagrange{RefQuadrilateral, 1}(), [0.7872983346207417 0.44364916731037085 … 0.05635083268962915 0.012701665379258308; 0.09999999999999994 0.44364916731037085 … 0.05635083268962915 0.09999999999999999; 0.012701665379258296 0.05635083268962912 … 0.44364916731037085 0.7872983346207417; 0.09999999999999994 0.05635083268962912 … 0.44364916731037085 0.09999999999999999], [0.7872983346207417 0.44364916731037085 … 0.05635083268962915 0.012701665379258308; 0.099999999999999

In [7]:
cond_vec = project_function_to_fem(fe, itp; space=:σ)
cond_vec .= min.(max.(cond_vec, 1e-6), 2.8)

3969-element SparseArrays.SparseVector{Float64, Int64} with 3969 stored entries:
  [1   ]  =  1.43257
  [2   ]  =  1.4845
  [3   ]  =  1.46419
  [4   ]  =  1.47182
  [5   ]  =  1.49761
  [6   ]  =  1.5537
  [7   ]  =  1.59288
          ⋮
  [3962]  =  0.391354
  [3963]  =  0.405863
  [3964]  =  0.443788
  [3965]  =  0.403251
  [3966]  =  0.383249
  [3967]  =  0.386418
  [3968]  =  0.377171
  [3969]  =  0.382861

In [8]:
eval_points = reshape(equidistant_grid(64), :)
ph = PointEvalHandler(grid, eval_points)

PointEvalHandler{Grid{2, Quadrilateral, Float64}, Float64}
  number of points: 4096
  Found corresponding cell for all points.

In [9]:
# G must be mean zero on the boundary (Neumann data integrates to 0);
# `down`/`up` batch over columns, so this normalizes all modes at once.
mean_zero_boundary(G_boundary) = G_boundary .- Statistics.mean(G_boundary, dims=1)

mean_zero_boundary (generic function with 1 method)

In [10]:
G_full = real_fourier_basis(8)
rhs_dict = Dict()
Threads.@threads for i in 2:253
    M = make_boundary(G_full[:, i],64)
    itp = interpolate_array_2D(M)
    rhs_dict[i-1] = assemble_rhs_func(fe, itp)
end
G = reduce(hcat, [rhs_dict[i] for i in 1:252])
G = fe.up(mean_zero_boundary(fe.down(G)))

4096×252 Matrix{Float64}:
  0.00280569   7.76464e-20  0.00280483  …   0.000906778  -2.62155e-17
  0.00280482   6.99479e-5   0.00280135     -0.000905793   4.23107e-5
  0.0          0.0          0.0             0.0           0.0
  0.00280482  -6.99479e-5   0.00280135     -0.000905793  -4.23107e-5
  0.00280221   0.000139852  0.0027909       0.00090284   -8.453e-5
  0.0          0.0          0.0         …   0.0           0.0
  0.00279785   0.00020967   0.00277351     -0.000897926   0.000126567
  0.0          0.0          0.0             0.0           0.0
  0.00279175   0.000279357  0.00274924      0.00089106   -0.00016833
  0.0          0.0          0.0             0.0           0.0
  ⋮                                     ⋱   ⋮            
 -0.00275007  -0.000555939  0.00258461      0.000821537  -0.000321897
 -0.00276308  -0.000487205  0.00263572     -0.000862979   0.000292759
 -0.00277436  -0.000418168  0.00268025      0.000871539  -0.000250676
 -0.00278392  -0.000348871  0.00271812  …  -

In [11]:
G_in = fe.up(mean_zero_boundary(fe.down(G) .+ σg .* randn(fe.m, 252)))
fbm_true = FerriteBlockMode(cond_vec, G_in, fe; block=false)

FerriteBlockMode(ModularEIT.BlockLAssembler(4, [0.25524077362757097 -0.06381019340689274 -0.12762038681378543 -0.06381019340689281; -0.06381019340689274 0.25524077362757147 -0.06381019340689284 -0.12762038681378587; -0.12762038681378543 -0.06381019340689284 0.2552407736275713 -0.063810193406893; -0.06381019340689281 -0.12762038681378587 -0.063810193406893 0.25524077362757164]), [1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], sparse([1, 2, 3, 4, 1, 2, 3, 4, 5, 6  …  4030, 4031, 4032, 4094, 4095, 4096, 4031, 4032, 4095, 4096], [1, 1, 1, 1, 2, 2, 2, 2, 2, 2  …  4095, 4095, 4095, 4095, 4095, 4095, 4096, 4096, 4096, 4096], [0.9550479532649173, -0.2387619883162296, -0.4775239766324585, -0.23876198831622916, -0.2387619883162296, 1.944714705659659, -0.4861786764149203, -0.4775239766324585, -0.24741668809868, -0.49483337619737067  …  -0.12572368871325723, -0.12667203776352012, -0.12762038681378587, -0.06286184435662999, 0.5066881510540862

In [12]:
F = copy(fbm_true.F) + σf * randn(size(fbm_true.F))
F = mean_zero_boundary(F)
GS = copy(fe.down(G))
F, G, Σ = svd_on_modes(F, GS)

# svd_on_modes already returns G projected onto the boundary basis (fe.m rows),
# so it must not be passed through fe.down again -- just re-normalize and lift.
F = mean_zero_boundary(F)
G = fe.up(mean_zero_boundary(G .+ σg .* randn(size(G))))

4096×252 Matrix{Float64}:
 -0.0190497  -0.0619877  0.0150928  …   1.18477e-9    1.06859e-15
 -0.0188694  -0.0611788  0.014748      -3.05586e-9   -3.21965e-15
  0.0         0.0        0.0            0.0           0.0
 -0.0188553  -0.0612277  0.0147593     -2.80307e-9    2.05391e-15
 -0.0187134  -0.0603682  0.0144132     -1.12718e-8    7.21645e-16
  0.0         0.0        0.0        …   0.0           0.0
 -0.018574   -0.0595176  0.0140715     -1.90311e-8    4.30211e-16
  0.0         0.0        0.0            0.0           0.0
 -0.0184498  -0.0586263  0.0137224     -2.80781e-8   -9.71445e-17
  0.0         0.0        0.0            0.0           0.0
  ⋮                                 ⋱   ⋮            
  0.125202    0.0615979  0.0608937     -5.52865e-8   -9.71445e-17
  0.132003    0.0679656  0.0664933     -4.64731e-8    3.19189e-16
  0.138363    0.0738781  0.0720917     -4.12731e-8   -2.22045e-16
  0.144383    0.079315   0.0779273  …  -3.37035e-8   -6.93889e-17
  0.151222    0.0853192  0.0

In [13]:
fbm = FerriteBlockMode(F, G, fe)

FerriteBlockMode(ModularEIT.BlockLAssembler(4, [0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0]), [1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], sparse([1, 2, 3, 4, 1, 2, 3, 4, 5, 6  …  4030, 4031, 4032, 4094, 4095, 4096, 4031, 4032, 4095, 4096], [1, 1, 1, 1, 2, 2, 2, 2, 2, 2  …  4095, 4095, 4095, 4095, 4095, 4095, 4096, 4096, 4096, 4096], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 4096, 4096), sparse([1, 2, 3, 4, 4097, 1, 2, 3, 4, 5  …  4087, 4088, 4089, 4090, 4091, 4092, 4093, 4094, 4095, 4096], [1, 1, 1, 1, 1, 2, 2, 2, 2, 2  …  4097, 4097, 4097, 4097, 4097, 4097, 4097, 4097, 4097, 4097], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], 4097, 4097), [-1.3651022150169072 -3.456669111873082 … -2.0215419130297565e-10 -4.776306262080345e-32; -1.352181826465278 -3.4115576140179664 … -1.2570364306

In [14]:
mean(G[:,1])

-4.336808689942018e-19

In [ ]:
f, ∂f =  create_block_f∂f(fbm, fe; nmodes=25, block=true, gn=false, maxiter = 1000)

In [ ]:
#prox_obj = create_proximal_gradient_step(f,∂f,ρ_obj, fe.n_σ)
prox_obj = create_prox_linesearch(f,∂f,ρ_obj)

#103 (generic function with 1 method)

In [18]:
fbm

FerriteBlockMode(ModularEIT.BlockLAssembler(4, [0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0]), [1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], sparse([1, 2, 3, 4, 1, 2, 3, 4, 5, 6  …  4030, 4031, 4032, 4094, 4095, 4096, 4031, 4032, 4095, 4096], [1, 1, 1, 1, 2, 2, 2, 2, 2, 2  …  4095, 4095, 4095, 4095, 4095, 4095, 4096, 4096, 4096, 4096], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 4096, 4096), sparse([1, 2, 3, 4, 4097, 1, 2, 3, 4, 5  …  4087, 4088, 4089, 4090, 4091, 4092, 4093, 4094, 4095, 4096], [1, 1, 1, 1, 1, 2, 2, 2, 2, 2  …  4097, 4097, 4097, 4097, 4097, 4097, 4097, 4097, 4097, 4097], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], 4097, 4097), [-1.3651022150169072 -3.456669111873082 … -2.0215419130297565e-10 -4.776306262080345e-32; -1.352181826465278 -3.4115576140179664 … -1.2570364306

In [19]:
σ₁ = ones(fe.n_σ)


3969-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 ⋮
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0

In [ ]:
σᵢ, err, _ = prox_obj(σ₁) 

In [ ]:
# Choose some starting sigma:

In [ ]:
σ_img = log.(reshape(evaluate_at_points(ph, fe.dh_σ, σᵢ), (64, 64)))

In [ ]:
σ_img = Gray.(((σ_img) .*0.5) .+0.5)

In [ ]:
#save("lbfgsb_8_unregularized1e-3.png", map(clamp01nan, σ_img))

In [ ]:
# cond_vec lives on the σ-space (fe.dh_σ), not the u-space (fe.dh) — must
# evaluate against fe.dh_σ to read the conductivity image back out.
cond_img = reshape(evaluate_at_points(ph, fe.dh_σ, cond_vec), (64, 64))
cond_img = Gray.(log.(cond_img) .*0.5 .+0.5)

In [ ]:
#save("original.png", map(clamp01nan, cond_img))